In [1]:
!pip install langchain langchain_core langchain-community pypdf pymupdf
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.6/335.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from langchain_core.documents import Document
doc=Document(page_content="this is my document.i am using to create RAG",
             metadata={"my_key":"my_value",
                       "source":"example.txt",
                       "author":"ashok"})
doc

Document(metadata={'my_key': 'my_value', 'source': 'example.txt', 'author': 'ashok'}, page_content='this is my document.i am using to create RAG')

In [3]:
 #creating a sample txt file
 import os
 os.makedirs("data/text_files",exist_ok=True)

In [ ]:
# ### textloader

# from langchain_community.document_loaders import TextLoader
# loader=TextLoader("data/text_files/sample1.txt")
# loader.load()


In [ ]:
# ### Directory Loader
# from langchain_community.document_loaders import DirectoryLoader

# ## load all the text files from the directory
# dir_loader=DirectoryLoader(
#     "data/text_files",
#     glob="**/*.txt", ## Pattern to match files
#     loader_cls= TextLoader, ##loader class to use
#     loader_kwargs={'encoding': 'utf-8'},
#     show_progress=False

# )

# documents=dir_loader.load()
# documents

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()
print("Loaded documents:", len(pdf_documents))


In [ ]:
!pip install langchain-text-splitters

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("data/text_files")

Found 4 PDF files to process

Processing: Dsa.pdf
  ✓ Loaded 112 pages

Processing: python basics.pdf
  ✓ Loaded 155 pages

Processing: MACHINE LEARNING(R17A0534).pdf
  ✓ Loaded 0 pages

Processing: psychology_explained.pdf
  ✓ Loaded 0 pages

Total documents loaded: 267


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'data/text_files/Dsa.pdf', 'file_path': 'data/text_files/Dsa.pdf', 'total_pages': 112, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20081218224633+10'00'", 'page': 0, 'source_file': 'Dsa.pdf', 'file_type': 'pdf'}, page_content='DSA\nD a t a  S t r u c t u r e s  a n d  A l g o r i t h m s\nAnnotated Reference with Examples\nGranville Barne!\nLuca Del Tongo'),
 Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'data/text_files/Dsa.pdf', 'file_path': 'data/text_files/Dsa.pdf', 'total_pages': 112, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate

In [ ]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1500,chunk_overlap=300):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 267 documents into 469 chunks

Example chunk:
Content: DSA
D a t a  S t r u c t u r e s  a n d  A l g o r i t h m s
Annotated Reference with Examples
Granville Barne!
Luca Del Tongo...
Metadata: {'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'data/text_files/Dsa.pdf', 'file_path': 'data/text_files/Dsa.pdf', 'total_pages': 112, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20081218224633+10'00'", 'page': 0, 'source_file': 'Dsa.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'data/text_files/Dsa.pdf', 'file_path': 'data/text_files/Dsa.pdf', 'total_pages': 112, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20081218224633+10'00'", 'page': 0, 'source_file': 'Dsa.pdf', 'file_type': 'pdf'}, page_content='DSA\nD a t a  S t r u c t u r e s  a n d  A l g o r i t h m s\nAnnotated Reference with Examples\nGranville Barne!\nLuca Del Tongo'),
 Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'data/text_files/Dsa.pdf', 'file_path': 'data/text_files/Dsa.pdf', 'total_pages': 112, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate

In [ ]:

!pip install -qU chromadb langchain-chroma langchain sentence-transformers
!pip install rfc3987
import numpy as np
import uuid
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_53610/2929466218.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 469 texts...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Generated embeddings with shape: (469, 384)
Adding 469 documents to vector store...
Successfully added 469 documents to vector store
Total documents in collection: 469


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,
)

def docu_chat(user_query):
    # 🔥 Directly use your retriever (no extra function needed)
    results = rag_retriever.retrieve(user_query, top_k=3)

    # 🔥 Build context
    context = ""
    source_docs = []

    for doc in results:
        context += doc["content"] + "\n\n"
        source_docs.append({
            "source": doc["metadata"].get("source_file", "unknown"),
            "page": doc["metadata"].get("page", "N/A"),
            "score": doc["similarity_score"]
        })

    # 🔥 System prompt
    system_message = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{context}
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_query}
    ]

    # 🔥 Call LLM
    response = model.invoke(messages)

    return {
        "answer": response.content,
        "source_documents": source_docs,
        "context_used": context
    }

In [ ]:
# result = docu_chat("how to delete node in linked list")

# print(result["answer"])
# print(result["source_documents"])

Retrieving documents for query: 'how to delete node in linked list'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Deleting a node from a linked list is straightforward, but there are several cases that need to be accounted for:

1.  The list is empty.
2.  The node to remove is the only node in the linked list.
3.  We are removing the head node.
4.  We are removing the tail node.
5.  The node to remove is somewhere in between the head and tail.
6.  The item to remove doesn’t exist in the linked list.

The context states that an algorithm exists to handle these cases to remove a node from anywhere within a list. If items are only ever removed from the front of the linked list, deletion becomes an O(1) operation.

The provided text describes the scenarios for deletion but does not provide the specific steps or algorithm for how to perform the deletion.
[{'source': 'Dsa.pdf', 'page': 21, 'score': 0.3664480447769165}, {'source': 'Dsa.pdf', 'page': 24, 'score': 0.09257686138153076}]


In [ ]:
import gradio as gr

# 🔥 Your existing function (unchanged)
def rag_chat_interface(query):
    if query.lower() == "exit":
        return "Goodbye 👋", ""

    result = docu_chat(query)

    answer = result["answer"]

    if result["source_documents"]:
        sources = "\n".join([f"- {src}" for src in result["source_documents"]])
    else:
        sources = "No sources found"

    return answer, sources


# 🎨 UI
with gr.Blocks() as demo:
    gr.Markdown("# 📄 RAG Chatbot")

    query_input = gr.Textbox(
        label="Ask your question",
        placeholder="Type your question here..."
    )

    answer_output = gr.Textbox(label="Answer")
    source_output = gr.Textbox(label=" Sources")

    submit_btn = gr.Button("Submit")

    submit_btn.click(
        fn=rag_chat_interface,
        inputs=query_input,
        outputs=[answer_output, source_output]
    )

# 🚀 Run
demo.launch()


Ask your question (type 'exit' to quit): how to delete node in linked list
Retrieving documents for query: 'how to delete node in linked list'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)

🧠 Answer:
Deleting a node from a linked list is straightforward, but there are several cases to account for. The provided text outlines these scenarios:

1.  The list is empty.
2.  The node to remove is the only node in the linked list.
3.  We are removing the head node.
4.  We are removing the tail node.
5.  The node to remove is somewhere in between the head and tail.
6.  The item to remove doesn't exist in the linked list.

The text states that an algorithm exists to handle these cases, allowing removal from anywhere within a list. It also mentions that if items are only ever removed from the head or tail, more concise algorithms can be created, with removal from the front being an O(1) operation.

However, the specific algorithm or steps for how to perform the deletion for each of these cases are not provided in the given context.

📄 Sources:
- {'source': 'Dsa.pdf', 'page': 21, 'score': 0.36644804477

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

🧠 Answer:
I don't know.

📄 Sources:
No sources found

Ask your question (type 'exit' to quit): what is dictionary and explian with example
Retrieving documents for query: 'what is dictionary and explian with example'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 12.453167585s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
# !pip install langchain-groq

# from langchain_groq import ChatGroq
# from langchain_core.prompts import PromptTemplate
# from langchain_core.messages import HumanMessage, SystemMessage


In [ ]:
# class GroqLLM:
#     def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
#         """
#         Initialize Groq LLM

#         Args:
#             model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
#             api_key: Groq API key (or set GROQ_API_KEY environment variable)
#         """
#         self.model_name = model_name
#         self.api_key = api_key or os.environ.get("GROQ_API_KEY")

#         if not self.api_key:
#             raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

#         self.llm = ChatGroq(
#             groq_api_key=self.api_key,
#             model_name=self.model_name,
#             temperature=0.1,
#             max_tokens=1024
#         )

#         print(f"Initialized Groq LLM with model: {self.model_name}")

#     def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
#         """
#         Generate response using retrieved context

#         Args:
#             query: User question
#             context: Retrieved document context
#             max_length: Maximum response length

#         Returns:
#             Generated response string
#         """

#         # Create prompt template
#         prompt_template = PromptTemplate(
#             input_variables=["context", "question"],
#             template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

# Context:
# {context}

# Question: {question}

# Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
#         )

#         # Format the prompt
#         formatted_prompt = prompt_template.format(context=context, question=query)

#         try:
#             # Generate response
#             messages = [HumanMessage(content=formatted_prompt)]
#             response = self.llm.invoke(messages)
#             return response.content

#         except Exception as e:
#             return f"Error generating response: {str(e)}"

#     def generate_response_simple(self, query: str, context: str) -> str:
#         """
#         Simple response generation without complex prompting

#         Args:
#             query: User question
#             context: Retrieved context

#         Returns:
#             Generated response
#         """
#         simple_prompt = f"""Based on this context: {context}

# Question: {query}

# Answer:"""

#         try:
#             messages = [HumanMessage(content=simple_prompt)]
#             response = self.llm.invoke(messages)
#             return response.content
#         except Exception as e:
#             return f"Error: {str(e)}"


In [ ]:
# # Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
# try:
#     groq_llm = GroqLLM(api_key="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")
#     print("Groq LLM initialized successfully!")
# except ValueError as e:
#     print(f"Warning: {e}")
#     print("Please set your GROQ_API_KEY environment variable to use the LLM.")
#     groq_llm = None

In [ ]:
# ### Simple RAG pipeline with Groq LLM
# from langchain_groq import ChatGroq

# ### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
# groq_api_key ="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"

# llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

# ## 2. Simple RAG function: retrieve context + generate response
# def rag_simple(query,retriever,llm,top_k=3):
#     ## retriever the context
#     results=retriever.retrieve(query,top_k=top_k)
#     context="\n\n".join([doc['content'] for doc in results]) if results else ""
#     if not context:
#         return "No relevant context found to answer the question."

#     ## generate the answwer using GROQ LLM
#     prompt=f"""Use the following context to answer the question concisely.
#         Context:
#         {context}

#         Question: {query}

#         Answer:"""

#     response=llm.invoke([prompt.format(context=context,query=query)])
#     return response.content

In [ ]:
# answer=rag_simple("what is linear regression",rag_retriever,llm)
# print(answer)